* Develop your own Machine Learning method for classification
  * Develop (train) at least two ML classifier on the extracted features (using
  pyOpenMS’ FeatureFinder algorithm) of the benchmark data-set (mzXML files)
  that can be used to classify two classes. You can define the two classes yourself.
  * Use two algorithms you have NOT used before (see e.g. here: https://scikitlearn.org/stable/supervised_learning.html#supervised-learning)
  * Use the Tidy Data Framework (see week 5) as structure for your pre-processed
  data (use binning!).
  * OPTIONAL PREPROCESSING: implement other pre-processing strategy to such as
smoothing and normalization.
* Evaluate and interpret the result
  * Evaluate your models using the usual performance measures (see e.g. week 5).
  * Perform a feature importance analysis and interpret the results

In [ ]:
!pip install pyopenms

     |████████████████████████████████| 40.0MB 105kB/s 


In [ ]:
from pyopenms import *
import pandas as pd
import sys
import csv

In [ ]:
ec08 = MSExperiment()
MzXMLFile().load("EC08.mzXML", ec08)

In [ ]:
ec15 = MSExperiment()
MzXMLFile().load("EC15.mzXML", ec15)

## Preprocessing 

* Smoothing

In [ ]:
gf = GaussFilter()
param = gf.getParameters()
param.setValue("gaussian_width",2.0) # needs wider width
gf.setParameters(param)
gf.filterExperiment(ec08)
gf.filterExperiment(ec15)

* Filtering by MS level

MS lv == 1 , because feature detection works with MS level 1 only

In [ ]:
exp_08 = MSExperiment()

for s in ec08:
  if s.getMSLevel() == 1:
    exp_08.addSpectrum(s)

MzMLFile().store("EC08.smt.MS.mzML", exp_08)

In [ ]:
exp_15 = MSExperiment()

for s in ec15:
  if s.getMSLevel() == 1:
    exp_15.addSpectrum(s)

MzMLFile().store("EC15.smt.MS.mzML", exp_15)

* Feature Detection

In [ ]:
# Load data
input_08 = MSExperiment()

MzMLFile().load("EC08.smt.MS.mzML", input_08)

input_08.updateRanges()

In [ ]:
input_15 = MSExperiment()

MzMLFile().load("EC15.smt.MS.mzML", input_15)

input_15.updateRanges()

In [ ]:
f8 = FeatureFinder()
f8.setLogType(LogType.CMD)

f15 = FeatureFinder()
f15.setLogType(LogType.CMD)

In [ ]:
# Run the feature finder
name1 = "centroided"
features1 = FeatureMap()
seeds1 = FeatureMap()
params1 = FeatureFinder().getParameters(name1)

name3 = "centroided"
features3 = FeatureMap()
seeds3 = FeatureMap()
params3 = FeatureFinder().getParameters(name3)

In [ ]:
f8.run(name1, input_08, features1, params1, seeds1)

In [ ]:
f15.run(name3, input_15, features3, params3, seeds3)

In [ ]:
features1.setUniqueIds()
fh1 = FeatureXMLFile()
fh1.store("output.feature_8_XML", features1)
print("EC08:", features1.size(), "features")

EC08: 77875 features


In [ ]:
features3.setUniqueIds()
fh3 = FeatureXMLFile()
fh3.store("output.feature_15_XML", features3)
print("EC15:", features3.size(), "features")

EC15: 72370 features


In [ ]:
help(features1)

In [ ]:
pr = features1[0].getMetaValue("PeptideRef")
print(pr)

None


## Format XML to Tidy

In [ ]:
def convert_to_row(first, run_id, keys, filename):
    peptide_ref = first.getMetaValue("PeptideRef")
   
    full_peptide_name = "" # don't  have
    protein_name = ""  # don't  have
    getChargeState = ""  # don't  have
    sequence = ""  # don't  have
    row = [
        first.getMetaValue("PeptideRef"),
        run_id,
        filename,
        first.getRT(),
        first.getUniqueId(),
        sequence,
        full_peptide_name,
        first.getCharge(),
        first.getMZ(),
        first.getIntensity(),
        protein_name
    ]

    for k in keys:
        row.append(first.getMetaValue(k))

    return row

def get_header(features):
    keys = []
    features[0].getKeys(keys)
    header = [
        "transition_group_id",
        "run_id",
        "filename",
        "RT",
        "id",
        "Sequence" ,
        "FullPeptideName",
        "Charge",
        "m/z",
        "Intensity",
        "ProteinName",
        ]
    header.extend(keys)
    return header

In [ ]:
#  load featureXML
features01 = FeatureMap()
fh = FileHandler()
fh.loadFeatures("output.feature_8_XML", features01)

# write TSV file
filename = "EC08.tsv"
fh = open(filename, "w")
wr = csv.writer(fh, delimiter='\t')
header = get_header(features01)
wr.writerow(header)
for f in features01:
    keys = []
    f.getKeys(keys)
    row = convert_to_row(f,filename,keys,filename)
    wr.writerow(row)
fh.close()  

In [ ]:
df = pd.read_csv("EC08.tsv", sep='\t')
df.head()

,transition_group_id,run_id,filename,RT,id,Sequence,FullPeptideName,Charge,m/z,Intensity,ProteinName,b'label',b'score_fit',b'score_correlation',b'FWHM',b'spectrum_index',b'spectrum_native_id'
0,NaN,EC08.tsv,EC08.tsv,5204.243655,11416192732915646490,NaN,NaN,4,982.815899,31995050.0,NaN,321251,0.812599,0.941835,23.946774,3024,scan=10545
1,NaN,EC08.tsv,EC08.tsv,5204.261942,14911396596348848003,NaN,NaN,4,982.769250,31871600.0,NaN,321247,0.789909,0.940066,23.793114,3024,scan=10545
2,NaN,EC08.tsv,EC08.tsv,5204.261079,7340105522523449647,NaN,NaN,4,982.775109,31863150.0,NaN,321249,0.792364,0.942022,23.794535,3024,scan=10545
3,NaN,EC08.tsv,EC08.tsv,5204.335200,14782519022896346546,NaN,NaN,4,982.757264,31732060.0,NaN,321242,0.766562,0.940915,23.560675,3024,scan=10545
4,NaN,EC08.tsv,EC08.tsv,5204.329244,2619653994644277784,NaN,NaN,4,982.915673,31552850.0,NaN,321220,0.710372,0.908830,23.946663,3024,scan=10545


In [ ]:
#  load featureXML
features03 = FeatureMap()
fh = FileHandler()
fh.loadFeatures("output.feature_15_XML", features03)

# write TSV file
filename = "EC15.tsv"
fh = open(filename, "w")
wr = csv.writer(fh, delimiter='\t')
header = get_header(features03)
wr.writerow(header)
for f in features03:
    keys = []
    f.getKeys(keys)
    row = convert_to_row(f,filename,keys,filename)
    wr.writerow(row)
fh.close()  

In [ ]:
df = pd.read_csv("EC15.tsv", sep='\t')
df.head()

,transition_group_id,run_id,filename,RT,id,Sequence,FullPeptideName,Charge,m/z,Intensity,ProteinName,b'label',b'score_fit',b'score_correlation',b'FWHM',b'spectrum_index',b'spectrum_native_id'
0,NaN,EC15.tsv,EC15.tsv,5109.240839,10684807191947887155,NaN,NaN,4,982.880517,32799230.0,NaN,286715,0.644538,0.914858,23.043669,3099,scan=10269
1,NaN,EC15.tsv,EC15.tsv,5109.253541,1115646367029217435,NaN,NaN,4,983.214367,32733150.0,NaN,286727,0.735312,0.983207,23.407570,3099,scan=10269
2,NaN,EC15.tsv,EC15.tsv,5109.207656,9014583044893410553,NaN,NaN,4,983.196789,32643940.0,NaN,286724,0.771029,0.983294,23.394093,3099,scan=10269
3,NaN,EC15.tsv,EC15.tsv,5109.281499,3567527937088341580,NaN,NaN,4,983.249524,32225450.0,NaN,286737,0.754991,0.984326,23.397121,3099,scan=10269
4,NaN,EC15.tsv,EC15.tsv,5109.289764,13408497517450000015,NaN,NaN,4,983.255383,32222030.0,NaN,286738,0.750434,0.984382,23.414234,3099,scan=10269
